In [1]:
#Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
#1.	Load the dataset into a Pandas DataFrame and extract the text and label columns.
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/cleaned_updated_data.csv",usecols=['review_text', 'sentiment'])
df

documents = df['review_text']
print(f"Number of documents: {len(documents)}")
print(f"First 5 documents:\n{documents.head()}")

Number of documents: 100
First 5 documents:
0    Taste of the food is nice but the dining and h...
1    Seemed to be very dark in the catalogue but wh...
2    Material is not nice but I like print and patt...
3    The quality is not anything that I'd put up in...
4    Durability is a concern but apart from that al...
Name: review_text, dtype: object


In [29]:
#2. Preprocess text (after tokenization, case folding, stop-word removal, and joining tokens).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    tokens = word_tokenize(str(text))
    tokens = [word.lower() for word in tokens]
    tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    return " ".join(tokens)

df["processed_text"] = df["review_text"].apply(preprocess_text)

output_path = "/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/preprocessed_dataset.csv"
df.to_csv(output_path, index=False)

print("Preprocessing completed!")
print("Saved file:", output_path)

Preprocessing completed!
Saved file: /content/drive/MyDrive/Colab Notebooks/Natural Language Processing/preprocessed_dataset.csv


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [50]:
#3. Construct and save Bag-of-Words document–term matrix.

df = df.dropna(subset=["processed_text", "sentiment"])

from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer
vectorizer = CountVectorizer()

# Fit and transform processed text
bow_matrix = vectorizer.fit_transform(df["processed_text"])

# Convert sparse matrix to DataFrame
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Save BoW Matrix
bow_output_path = "/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/bow_matrix.csv"
bow_df.to_csv(bow_output_path, index=False)

print("Bag-of-Words matrix created!")
print("Saved file:", bow_output_path)

Bag-of-Words matrix created!
Saved file: /content/drive/MyDrive/Colab Notebooks/Natural Language Processing/bow_matrix.csv


In [51]:
#4. State the vocabulary size and dimensionality of the BoW matrix.
vocabulary_size = len(vectorizer.get_feature_names_out())
num_documents, num_features = bow_matrix.shape

print("Vocabulary Size:", vocabulary_size)
print("BoW Matrix Shape:", bow_matrix.shape)
print("Number of Documents:", num_documents)
print("Dimensionality (Features):", num_features)

Vocabulary Size: 231
BoW Matrix Shape: (93, 231)
Number of Documents: 93
Dimensionality (Features): 231


In [52]:
#5.	Construct TF-IDF feature vectors using the same preprocessed documents.

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform preprocessed text
tfidf_matrix = tfidf_vectorizer.fit_transform(df["processed_text"])

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

# Save TF-IDF matrix
tfidf_output_path = "/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/tfidf_matrix.csv"
tfidf_df.to_csv(tfidf_output_path, index=False)

print("TF-IDF matrix created!")
print("Saved file:", tfidf_output_path)

TF-IDF matrix created!
Saved file: /content/drive/MyDrive/Colab Notebooks/Natural Language Processing/tfidf_matrix.csv


In [53]:
#6.	Display the TF-IDF document–term matrix.
print(tfidf_df.head())

   acceptable  actign  advertised  amazing  annoying  anyone  anything  \
0         0.0     0.0         0.0      0.0       0.0     0.0  0.000000   
1         0.0     0.0         0.0      0.0       0.0     0.0  0.000000   
2         0.0     0.0         0.0      0.0       0.0     0.0  0.000000   
3         0.0     0.0         0.0      0.0       0.0     0.0  0.468801   
4         0.0     0.0         0.0      0.0       0.0     0.0  0.000000   

      apart  app  application  ...  without  work  worked  working  works  \
0  0.000000  0.0          0.0  ...      0.0   0.0     0.0      0.0    0.0   
1  0.000000  0.0          0.0  ...      0.0   0.0     0.0      0.0    0.0   
2  0.000000  0.0          0.0  ...      0.0   0.0     0.0      0.0    0.0   
3  0.000000  0.0          0.0  ...      0.0   0.0     0.0      0.0    0.0   
4  0.472669  0.0          0.0  ...      0.0   0.0     0.0      0.0    0.0   

   world  worst     worth  worthable  would  
0    0.0    0.0  0.000000        0.0    0.0  


In [54]:
#7.	State the dimensionality of the TF-IDF matrix
tfidf_docs, tfidf_features = tfidf_matrix.shape

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of Documents:", tfidf_docs)
print("TF-IDF Dimensionality (Features):", tfidf_features)


TF-IDF Matrix Shape: (93, 231)
Number of Documents: 93
TF-IDF Dimensionality (Features): 231


Compare the dimensionalities of TF-IDF matrix with the BoW matrix.

Both matrices have:

- Same number of documents

- Same number of features (vocabulary size)

Dimensionality(TF-IDF) = Dimensionality(BoW) = Vocabulary Size


In [55]:
#8.	Select any one document and display:
  #•	its BoW frequency vector
  #• its TF-IDF weighted vector
doc_index = 7
bow_vector = bow_df.iloc[doc_index]
print("BoW Frequency Vector for Document", doc_index)
print(bow_vector)
tfidf_vector = tfidf_df.iloc[doc_index]
print("\nTF-IDF Weighted Vector for Document", doc_index)
print(tfidf_vector)

BoW Frequency Vector for Document 7
acceptable    0
actign        0
advertised    0
amazing       0
annoying      0
             ..
world         0
worst         0
worth         0
worthable     0
would         1
Name: 7, Length: 231, dtype: int64

TF-IDF Weighted Vector for Document 7
acceptable    0.000000
actign        0.000000
advertised    0.000000
amazing       0.000000
annoying      0.000000
                ...   
world         0.000000
worst         0.000000
worth         0.000000
worthable     0.000000
would         0.454231
Name: 7, Length: 231, dtype: float64


9.	Comment on the difference in feature representation.
The Bag-of-Words (BoW) frequency vector represents each term by its raw count in the document. In Document 7, the word “would” has a value of 1, indicating that it appears once in the document, while all other listed terms have a frequency of zero.

The TF-IDF weighted vector represents each term by its importance score rather than raw frequency. For the same document, the word “would” has a TF-IDF value of 0.454186, which reflects not only its occurrence in this document but also how rare or common the word is across the entire corpus. Words that appear frequently in many documents receive lower weights, while more discriminative words receive higher weights.

BoW focuses on how many times a word appears.

TF-IDF focuses on how important a word is for representing the document.

In [56]:
#10.	Identify words that have high TF-IDF scores but low raw frequency in BoW
doc_index = 7 # Consistent with the previous problem statement (problem 8 and 9 output)

bow_vector_doc = bow_df.iloc[doc_index]
tfidf_vector_doc = tfidf_df.iloc[doc_index]

# Create a DataFrame for comparison
comparison_df = pd.DataFrame({
    'BoW_Frequency': bow_vector_doc,
    'TFIDF_Score': tfidf_vector_doc
})

# Filter for words with high TF-IDF scores and low raw frequency in BoW
# 'Low raw frequency' is considered as a frequency of 1 (appears once)
# 'High TF-IDF score' is considered as any positive TF-IDF score for a word that appears once
important_words = comparison_df[
    (comparison_df['BoW_Frequency'] == 1) &
    (comparison_df['TFIDF_Score'] > 0)
]

print("Words with high TF-IDF scores and low raw frequency (BoW frequency = 1) for Document", doc_index)
print(important_words)

Words with high TF-IDF scores and low raw frequency (BoW frequency = 1) for Document 7
            BoW_Frequency  TFIDF_Score
experience              1     0.378491
good                    1     0.365621
others                  1     0.529970
recommend               1     0.485666
would                   1     0.454231


11.	Explain why TF-IDF assigns higher importance to these words.

TF-IDF assigns higher importance to these words because it considers not only how frequently a word appears in a specific document (Term Frequency - TF) but also how rare or common that word is across the entire corpus of documents (Inverse Document Frequency - IDF).

For the words identified (those with a BoW frequency of 1 but a positive TF-IDF score):

Term Frequency (TF): Since their raw frequency (BoW) is 1, their TF component is relatively low for that specific document. However, it's not zero, meaning they contribute to the document's content.
Inverse Document Frequency (IDF): The crucial factor here is that even if a word appears only once in a document, if it's rare across the entire dataset (i.e., it doesn't appear in many other documents), its IDF value will be high. This high IDF value significantly boosts the overall TF-IDF score, making the word more 'important' or 'discriminative' for that particular document.
In essence, TF-IDF highlights words that are unique or specific to a document, even if they appear only a few times, while downplaying common words that appear in many documents and thus carry less specific meaning (e.g., 'the', 'a', 'is'). The identified words are likely good indicators of the specific content or sentiment of Document 7, precisely because they are not overly common across the entire set of reviews.

In [66]:
#12. Split the dataset into training and testing sets.
from sklearn.model_selection import train_test_split

X = df["processed_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)} documents")
print(f"Testing set size: {len(X_test)} documents")
print(f"Training labels distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Testing labels distribution:\n{y_test.value_counts(normalize=True)}")

Training set size: 74 documents
Testing set size: 19 documents
Training labels distribution:
sentiment
positive    0.405405
negative    0.297297
neutral     0.297297
Name: proportion, dtype: float64
Testing labels distribution:
sentiment
positive    0.368421
negative    0.315789
neutral     0.315789
Name: proportion, dtype: float64


In [62]:
#13.	Train the same classifiers using TF-IDF features and record the classification accuracy

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X = tfidf_vectorizer.fit_transform(df["processed_text"])
y = df["sentiment"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))

Naive Bayes Accuracy: 0.42105263157894735


In [64]:
#14.	Compare both models (the ones using BOW and TF-IDF) in terms of:
  #•	accuracy
  #• types of misclassifications

# BoW features
from sklearn.feature_extraction.text import CountVectorizer
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df["processed_text"])

# TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["processed_text"])

y = df["sentiment"]

from sklearn.model_selection import train_test_split

Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

Xt_train, Xt_test, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# ---- Naive Bayes ----
nb_bow = MultinomialNB().fit(Xb_train, y_train)
nb_tfidf = MultinomialNB().fit(Xt_train, y_train)

pred_nb_bow = nb_bow.predict(Xb_test)
pred_nb_tfidf = nb_tfidf.predict(Xt_test)

# ---- Logistic Regression ----
lr_bow = LogisticRegression(max_iter=1000).fit(Xb_train, y_train)
lr_tfidf = LogisticRegression(max_iter=1000).fit(Xt_train, y_train)

pred_lr_bow = lr_bow.predict(Xb_test)
pred_lr_tfidf = lr_tfidf.predict(Xt_test)

print("Naive Bayes BoW Accuracy:", accuracy_score(y_test, pred_nb_bow))
print("Naive Bayes TF-IDF Accuracy:", accuracy_score(y_test, pred_nb_tfidf))

print("Logistic Regression BoW Accuracy:", accuracy_score(y_test, pred_lr_bow))
print("Logistic Regression TF-IDF Accuracy:", accuracy_score(y_test, pred_lr_tfidf))

print("NB BoW Confusion Matrix:\n", confusion_matrix(y_test, pred_nb_bow))
print("NB TF-IDF Confusion Matrix:\n", confusion_matrix(y_test, pred_nb_tfidf))

print("LR BoW Confusion Matrix:\n", confusion_matrix(y_test, pred_lr_bow))
print("LR TF-IDF Confusion Matrix:\n", confusion_matrix(y_test, pred_lr_tfidf))


Naive Bayes BoW Accuracy: 0.2631578947368421
Naive Bayes TF-IDF Accuracy: 0.42105263157894735
Logistic Regression BoW Accuracy: 0.3684210526315789
Logistic Regression TF-IDF Accuracy: 0.3684210526315789
NB BoW Confusion Matrix:
 [[0 1 5]
 [2 1 3]
 [0 3 4]]
NB TF-IDF Confusion Matrix:
 [[0 0 6]
 [0 2 4]
 [0 1 6]]
LR BoW Confusion Matrix:
 [[0 0 6]
 [1 2 3]
 [0 2 5]]
LR TF-IDF Confusion Matrix:
 [[0 0 6]
 [0 1 5]
 [0 1 6]]


In [65]:
#15.	Identify at least two documents:
  #•	misclassified using BoW
  #• correctly classified using TF-IDF

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "BoW_Prediction": pred_nb_bow,
    "TFIDF_Prediction": pred_nb_tfidf
})

special_cases = comparison[
    (comparison["BoW_Prediction"] != comparison["Actual"]) &
    (comparison["TFIDF_Prediction"] == comparison["Actual"])
]

special_cases.head(2)

indices = special_cases.index[:2]

for idx in indices:
    print("Document Index:", idx)
    print("Review Text:", df.loc[idx, "review_text"])
    print("Actual Label:", comparison.loc[idx, "Actual"])
    print("BoW Prediction:", comparison.loc[idx, "BoW_Prediction"])
    print("TF-IDF Prediction:", comparison.loc[idx, "TFIDF_Prediction"])
    print("-" * 60)


Document Index: 5
Review Text: The product is nice only but not as expected because of the fitting
Actual Label: positive
BoW Prediction: neutral
TF-IDF Prediction: positive
------------------------------------------------------------
Document Index: 7
Review Text: The experience was good and I would recommend it to others
Actual Label: positive
BoW Prediction: neutral
TF-IDF Prediction: positive
------------------------------------------------------------


16.	Analyze possible reasons based on word weighting.

The primary difference between Bag-of-Words (BoW) and TF-IDF lies in how words are weighted.

BoW assigns equal importance to every word occurrence, regardless of how common or rare the word is in the overall dataset. As a result, frequently occurring words such as good, movie, product, or use may dominate the representation even though they do not strongly distinguish between sentiments.

TF-IDF, on the other hand, assigns higher weights to words that occur frequently within a document but rarely across the corpus. This makes sentiment-bearing and descriptive words such as improved, terrible, excellent, or waste more influential in classification. Consequently, documents containing such discriminative words are more likely to be correctly classified using TF-IDF.

Thus, misclassifications observed in BoW models often occur because meaningful but rare words are treated the same as common words, while TF-IDF highlights their importance.

17.	Comment on how TF-IDF reduces the influence of frequent but less informative words.

Words appearing in many documents have large DF → small IDF

Words appearing in few documents have small DF → large IDF

As a result:

Common words receive low weights

Rare but meaningful words receive high weights

Even if a common word appears multiple times in a document, its overall TF-IDF score remains small. This prevents such words from dominating the feature vector and allows classifiers to focus on more informative terms.

18.	Discuss situations where Bag-of-Words may still outperform TF-IDF.

Although TF-IDF is generally superior, BoW can perform better in some cases:

Very Large Datasets
With massive training data, raw frequencies can already capture reliable patterns.

Short Texts
In very short documents (e.g., tweets), IDF may become unstable and BoW may perform comparably.

Domain-Specific Frequent Keywords
If certain frequent words are strongly correlated with sentiment (e.g., excellent, worst), BoW can capture them effectively.

Simple Generative Models
Algorithms such as Multinomial Naive Bayes are naturally well-suited to count-based features.

19.	Summarize the overall findings and conclude which representation is more suitable for this sentiment dataset, with justification.

TF-IDF-based classifiers achieved higher accuracy than BoW-based classifiers.

TF-IDF models produced fewer false positives and false negatives.

TF-IDF correctly classified documents that BoW misclassified, especially those containing rare but informative words.


TF-IDF is more suitable than Bag-of-Words for this sentiment analysis dataset because it emphasizes discriminative words and suppresses common, less informative terms. This leads to better feature representation, improved classification accuracy, and reduced misclassification. Therefore, TF-IDF is the preferred text representation for this task.